# Pairs fOU 01 — Stage A: the thesis pipeline at 5 minutes on crypto perpetuals

Protocol: `discussions/Pairs_fOU/001_claude_stage_a_preregistration.md`. Source: García Petrov (2026), *Fractional Ornstein–Uhlenbeck Statistical Arbitrage with Synthetic Equity Options*.

Rolling 7-day formation (Engle–Granger 1%, $H<0.5$ by change-of-frequency, fOU moments, structural $T_{70}$), conditional first-passage $T_{70}\le 48$ bars from the last 60 inferred fGn innovations, $|Z|\ge1.5$, next-bar execution, exit at convergence or $T_{70}$. Perpetual legs, 12 bp round trip of gross notional. Benchmarks: OU ($H=0.5$) and a plain z-score rule.

In [ ]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import fou_pairs as fp
importlib.reload(fp);assert fp.VERSION=='PF01-0.1.0'
spec=fp.Spec()
SYNTHETIC=os.getenv('PF01_SYNTHETIC','0')=='1'
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='pf01-');OUT=Path(tmp.name)/'01';spec=fp.Spec(paths=1000,structural_paths=500,n_boot=500)
else:
    OUT=ROOT/'notebooks'/'Pairs_fOU'/'01'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_fou_pairs.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
print(chk.stdout[-1200:]);print(chk.stderr[-600:]);assert chk.returncode==0,'tests failed'
print(json.dumps(fp.registration(spec),indent=1))

In [ ]:
if SYNTHETIC:
    P=fp.synthetic_prices(n_days=14,seed=7,kappa=0.04,H=0.4)
    PAIRS=[('AAA','BBB')]
else:
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
    S={a:fp.log_close_5m(pd.read_parquet(BIN/f'{a}USDT_um_1m.parquet',columns=['open_time','close'])) for a in spec.assets}
    P=fp.panel(S);PAIRS=fp.pairs(spec.assets)
print('panel',P.shape,P.index.min(),'->',P.index.max(),'| pairs',PAIRS)
print('complete-bar share:',P.notna().mean().round(4).to_dict())

## Backtest (fOU, OU, z-score) — formation screening shared across modes

In [ ]:
T0=time.time();cache={};TR=[];FC=[];FI=[]
for (y,x) in PAIRS:
    for mode in ('fou','ou','zscore'):
        tr,fc,fi=fp.run_pair(P,y,x,spec,mode=mode,screen_cache=cache)
        TR.append(tr);FC.append(fc);FI.append(fi)
        print(f'{y}-{x} {mode:6s} trades={len(tr):5d} forecasts={len(fc):6d} eligible days={int(fi.eligible.sum()) if len(fi) else 0:4d} | {time.time()-T0:7.0f}s',flush=True)
trades=pd.concat([t for t in TR if len(t)],ignore_index=True) if any(len(t) for t in TR) else pd.DataFrame()
fcs=pd.concat([f for f in FC if len(f)],ignore_index=True) if any(len(f) for f in FC) else pd.DataFrame()
fits=pd.concat(FI,ignore_index=True)

In [ ]:
print('Formation funnel (pair-days):')
fu=fits.groupby(['mode','reason']).size().unstack(fill_value=0);display(fu)
f1=fits[fits['mode']=='fou']
print('fOU eligible pair-days by pair:');display(f1.groupby('pair').agg(days=('day','size'),coint=('coint_ok','mean'),eligible=('eligible','mean'),
      H_median=('H','median'),half_life_bars=('half_life_bars','median')).round(3))

In [ ]:
TS=pd.Timestamp(spec.test_start,tz='UTC')
if len(trades):
    trades['period']=np.where(trades.entry_bar>=TS,'test','dev')
rows=[]
for mode in ('fou','ou','zscore'):
    for per in ('dev','test'):
        t=trades[(trades['mode']==mode)&(trades.period==per)] if len(trades) else trades
        rows.append(fp.trade_summary(t,spec,f'{mode}|{per}|all'))
        for pr,g in (t.groupby('pair') if len(t) else []):
            rows.append(fp.trade_summary(g,spec,f'{mode}|{per}|{pr}'))
summ=pd.DataFrame(rows);display(summ.round(3))
cal=[]
for mode in ('fou','ou'):
    for per in ('dev','test'):
        f=fcs[(fcs['mode']==mode)] if len(fcs) else fcs
        if len(f):f=f[(f.signal_bar>=TS)==(per=='test')]
        cal.append({'mode':mode,'period':per,**fp.calibration(f)})
cal=pd.DataFrame(cal);print('Convergence calibration (thesis: 70.3% predicted vs 33.1% realised):');display(cal.round(4))

In [ ]:
dec=fp.decision(summ,synthetic=SYNTHETIC);print(json.dumps(dec,indent=2,default=float))
if len(trades):trades.to_csv(OUT/'01_trades.csv',index=False,float_format='%.6g')
fits.drop(columns=[c for c in ('resid',) if c in fits]).to_csv(OUT/'01_formation.csv',index=False,float_format='%.6g')
summ.to_csv(OUT/'01_summary.csv',index=False);cal.to_csv(OUT/'01_calibration.csv',index=False)
(OUT/'01_decision.json').write_text(json.dumps(dec,indent=2,default=float),encoding='utf-8')
(OUT/'01_manifest.json').write_text(json.dumps({'registration':fp.registration(spec),'synthetic':SYNTHETIC,
   'panel':[str(P.index.min()),str(P.index.max())],'runtime_s':time.time()-T0},indent=2,default=str),encoding='utf-8')
print('saved',sorted(p.name for p in OUT.iterdir()))